# Graph ML Capabilities

This notebook shows how to run graph analytics, PageRank, community detection, node embeddings, and link prediction on a constructed property graph.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next((p for p in Path.cwd().parents if (p / 'src').exists()), Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print(f'Repo root: {ROOT}')

In [ ]:
from kg_construction.application.graph_ml import graph_ml_workflow_config_from_dict, run_graph_ml_capabilities
from kg_construction.domain.graph_objects import PropertyEdge, PropertyGraph, PropertyNode

property_graph = PropertyGraph(
    nodes=[
        PropertyNode(node_id="svc:recommendation", labels=["Entity", "System"], properties={"name": "Recommendation API"}),
        PropertyNode(node_id="incident:1042", labels=["Entity", "Incident"], properties={"name": "INCIDENT-1042"}),
        PropertyNode(node_id="team:platform", labels=["Entity", "Team"], properties={"name": "Platform Team"}),
        PropertyNode(node_id="svc:gateway", labels=["Entity", "System"], properties={"name": "Gateway API"}),
    ],
    edges=[
        PropertyEdge(edge_id="e1", source_id="incident:1042", target_id="svc:recommendation", type="AFFECTED"),
        PropertyEdge(edge_id="e2", source_id="team:platform", target_id="svc:recommendation", type="OWNS"),
        PropertyEdge(edge_id="e3", source_id="svc:gateway", target_id="svc:recommendation", type="CALLS"),
    ],
    metadata={"notebook": "06_graph_ml_capabilities"},
)

property_graph.metadata

In [ ]:
config = graph_ml_workflow_config_from_dict(
    {
        "graph_ml": {
            "embeddings": {"method": "spectral", "dimensions": 6},
            "link_prediction": {"method": "hybrid", "max_predictions": 5, "add_edges": False},
        }
    }
)

enriched_graph = run_graph_ml_capabilities(property_graph, config=config)
enriched_graph.metadata["graph_ml_workflow"]

In [ ]:
node_rows = [
    {
        "node_id": node.node_id,
        "name": node.properties.get("name"),
        "degree": node.properties.get("degree"),
        "pagerank": node.properties.get("pagerank"),
        "community_id": node.properties.get("community_id"),
        "component_id": node.properties.get("component_id"),
        "embedding_preview": node.properties.get("node_embedding", [])[:3],
    }
    for node in enriched_graph.nodes
]
node_rows

In [ ]:
enriched_graph.metadata["link_prediction"]["predictions"]